# CS-323 Artificial Intelligence — Complex Engineering Problem
## Intelligent Vacuum-Cleaning Robot

**Group Members:**
| Roll No. |     Name       |
|----------|----------------|
| CS24105  | Sabika Abbas   |
| CS24110  | ALishba Javed  |
| CS24114  | Bushra Khalid  |
| CS24XXX  | Mariam Siddiqa |

---

### What this notebook does
1. Loads a grid environment from **`Environment.txt`**.
2. Loads scheduling constraints from **`Schedule.txt`**.
3. Uses **CSP (backtracking + forward checking)** to assign each dirty location to a time slot.
4. Uses **graph-based search (BFS and A*)** to navigate the robot through the cleaning order and back to the charger.
5. Reports the schedule, paths, cost, nodes expanded, and final grid.

### How to test ANY new case (no code changes needed)
- Put your files in the same folder as this notebook.
- Name them `Environment_*.txt` and `Schedule_*.txt` (e.g., `Environment_Test.txt` + `Schedule_Test.txt`).
- Run **Cell 12 (AUTO-DISCOVER & RUN ALL)** — it will find and run every pair automatically.

Or, to run one specific pair, just edit the two filenames in **Cell 3 (CONFIG)** and run Cell 11.

## 0. Imports & Configuration
Everything teacher might want to tweak is in ONE place (the CONFIG block below).

In [73]:
# ============================================================
# IMPORTS
# ============================================================
import ast
import os
import glob
import copy
import heapq
from collections import deque

# ============================================================
# CONFIG  —  change ONLY these
# ============================================================
DATA_FOLDER    = "inputs"              # <-- Folder ka naam yahan likhein

ENV_FILE       = os.path.join(DATA_FOLDER, "Environment.txt")
SCHEDULE_FILE  = os.path.join(DATA_FOLDER, "Schedule.txt")
ALGORITHM      = "A*"                  
VERBOSE        = True                  

print("Config loaded. Will run:", ENV_FILE, "+", SCHEDULE_FILE, "using", ALGORITHM)

Config loaded. Will run: inputs\Environment.txt + inputs\Schedule.txt using A*


## 1. Input Loading

The loaders are **generic** — they consume the file line-by-line with a pointer,
so they work for **any valid file** (0 or 6 obstacles, any number of dirty cells,
any number of slots, any number of constraints).

In [74]:
def load_environment(filepath):
    """
    Loads an environment file. Returns (grid, charging_station, dirty_locations).

    Format (per line):
        rows,cols
        num_obstacles
        (r,c),(r,c),...         (only if num_obstacles > 0)
        num_dirty
        (r,c),(r,c),...         (only if num_dirty > 0)
        (r,c)                   (charging station)
    """
    with open(filepath, 'r') as f:
        lines = [ln.strip() for ln in f if ln.strip()]

    i = 0
    rows, cols = map(int, lines[i].split(',')); i += 1
    grid = [['.' for _ in range(cols)] for _ in range(rows)]

    # Obstacles
    num_obstacles = int(lines[i]); i += 1
    if num_obstacles > 0:
        obstacles = ast.literal_eval(f"[{lines[i]}]"); i += 1
        for r, c in obstacles:
            grid[r][c] = 'X'

    # Dirty locations (D1, D2, ... assigned by order of appearance)
    num_dirty = int(lines[i]); i += 1
    dirty = []
    if num_dirty > 0:
        dirty = ast.literal_eval(f"[{lines[i]}]"); i += 1
        for k, (r, c) in enumerate(dirty):
            grid[r][c] = f"D{k+1}"

    # Charging station
    cs = ast.literal_eval(lines[i])
    grid[cs[0]][cs[1]] = 'C'

    return grid, cs, dirty

In [75]:
def load_schedule(filepath):
    """
    Loads a scheduling file. Returns (slots, precedence, latest_slot).

    Format (per line):
        num_slots
        num_precedence
        (u,v)   x num_precedence
        num_latest
        (task,slot)   x num_latest
    """
    with open(filepath, 'r') as f:
        lines = [ln.strip() for ln in f if ln.strip()]

    i = 0
    num_slots = int(lines[i]); i += 1
    slots = [f"T{k+1}" for k in range(num_slots)]

    num_prec = int(lines[i]); i += 1
    precedence = []
    for _ in range(num_prec):
        u, v = [s.strip() for s in lines[i].replace('(', '').replace(')', '').split(',')]
        precedence.append((u, v)); i += 1

    num_latest = int(lines[i]); i += 1
    latest_slot = {}
    for _ in range(num_latest):
        task, slot = [s.strip() for s in lines[i].replace('(', '').replace(')', '').split(',')]
        latest_slot[task] = slot; i += 1

    return slots, precedence, latest_slot


def slot_to_int(slot):
    """'T3' -> 3"""
    return int(slot[1:])

## 2. Graph-Based Search

Implement two algorithms:
- **BFS** — uninformed, complete, optimal (unit cost).
- **A\*** — informed, complete, optimal (with admissible Manhattan heuristic).

Problem formulation:
- **State:** (row, col)
- **Actions:** UP, DOWN, LEFT, RIGHT (no diagonals)
- **Transition:** move to neighbour if within grid and not `X`
- **Step cost:** 1
- **Heuristic:** Manhattan distance `|r - r_goal| + |c - c_goal|` — admissible because each move changes Manhattan distance by at most 1.

In [76]:
def get_successors(state, grid):
    """Returns [(next_state, action, step_cost), ...] for valid moves."""
    r, c = state
    rows, cols = len(grid), len(grid[0])
    moves = [(-1, 0, 'UP'), (1, 0, 'DOWN'), (0, -1, 'LEFT'), (0, 1, 'RIGHT')]
    out = []
    for dr, dc, action in moves:
        nr, nc = r + dr, c + dc
        if 0 <= nr < rows and 0 <= nc < cols and grid[nr][nc] != 'X':
            out.append(((nr, nc), action, 1))
    return out


def goal_test(state, goal_state):
    return state == goal_state


def path_cost(path):
    """Cost = number of moves = len(path) - 1. Empty path -> 0."""
    if not path:
        return 0
    return len(path) - 1


def heuristic(state, goal_state):
    """Manhattan distance — admissible for 4-directional movement."""
    return abs(state[0] - goal_state[0]) + abs(state[1] - goal_state[1])


def validate_path(path, grid):
    """Check that consecutive states are adjacent and never on an obstacle."""
    if not path:
        return True
    for i in range(len(path) - 1):
        r1, c1 = path[i]
        r2, c2 = path[i+1]
        if abs(r1 - r2) + abs(c1 - c2) != 1:
            print(f"  [INVALID] Non-adjacent move {path[i]} -> {path[i+1]}")
            return False
        if grid[r2][c2] == 'X':
            print(f"  [INVALID] Path enters obstacle at {path[i+1]}")
            return False
    return True

In [77]:
def bfs_search(grid, start, goal):
    """Returns (path, cost, nodes_expanded)."""
    if goal_test(start, goal):
        return [start], 0, 0

    frontier = deque([start])
    came_from = {start: None}
    nodes_expanded = 0

    while frontier:
        current = frontier.popleft()
        nodes_expanded += 1

        if goal_test(current, goal):
            path, node = [], current
            while node is not None:
                path.append(node); node = came_from[node]
            path.reverse()
            return path, path_cost(path), nodes_expanded

        for nxt, _, _ in get_successors(current, grid):
            if nxt not in came_from:
                frontier.append(nxt)
                came_from[nxt] = current

    return None, 0, nodes_expanded


def a_star_search(grid, start, goal):
    """Returns (path, cost, nodes_expanded)."""
    if goal_test(start, goal):
        return [start], 0, 0

    frontier = [(heuristic(start, goal), 0, start)]
    came_from = {start: None}
    g_costs = {start: 0}
    nodes_expanded = 0

    while frontier:
        f, g_cur, current = heapq.heappop(frontier)
        nodes_expanded += 1

        if goal_test(current, goal):
            path, node = [], current
            while node is not None:
                path.append(node); node = came_from[node]
            path.reverse()
            return path, path_cost(path), nodes_expanded

        for nxt, _, step in get_successors(current, grid):
            new_g = g_cur + step
            if nxt not in g_costs or new_g < g_costs[nxt]:
                g_costs[nxt] = new_g
                heapq.heappush(frontier, (new_g + heuristic(nxt, goal), new_g, nxt))
                came_from[nxt] = current

    return None, 0, nodes_expanded


def search(grid, start, goal, algorithm="A*"):
    """Dispatcher: 'BFS' or 'A*'."""
    alg = algorithm.upper().replace(" ", "")
    if alg == "BFS":
        return bfs_search(grid, start, goal)
    if alg in ("A*", "ASTAR", "A-STAR"):
        return a_star_search(grid, start, goal)
    raise ValueError(f"Unknown algorithm: {algorithm}")

## 3. Constraint Satisfaction Problem

- **Variables:** D1, D2, … (one per dirty location, loaded from Environment.txt)
- **Domains:** {T1, T2, …, Tn} (from Schedule.txt)
- **Constraints:**
  - Precedence (D_u before D_v)
  - Latest slot (D_x ≤ T_k)
  - Slot exclusivity (all-different)

Implemented:
- `is_consistent` — checks partial-constraint violations
- `forward_check` — prunes incompatible values; returns None if any domain empties
- `select_unassigned_variable` — MRV + degree heuristic tie-breaker
- `order_domain_values` — LCV (least constraining value)
- `backtracking_search` — with forward checking

In [78]:
def is_consistent(variable, value, assignment, precedence, latest_slot):
    # 1. Slot exclusivity
    if value in assignment.values():
        return False
    # 2. Latest slot
    if variable in latest_slot and slot_to_int(value) > slot_to_int(latest_slot[variable]):
        return False
    # 3. Precedence
    for u, v in precedence:
        if v == variable and u in assignment:
            if slot_to_int(assignment[u]) >= slot_to_int(value):
                return False
        if u == variable and v in assignment:
            if slot_to_int(value) >= slot_to_int(assignment[v]):
                return False
    return True


def forward_check(variable, value, assignment, domains, precedence, latest_slot):
    new_domains = copy.deepcopy(domains)
    for other in domains:
        if other in assignment:
            continue
        # Slot exclusivity
        if value in new_domains[other]:
            new_domains[other].remove(value)
        # Latest slot
        if other in latest_slot:
            new_domains[other] = [s for s in new_domains[other]
                                  if slot_to_int(s) <= slot_to_int(latest_slot[other])]
        # Precedence
        for u, v in precedence:
            if variable == u and other == v:
                new_domains[other] = [s for s in new_domains[other]
                                      if slot_to_int(s) > slot_to_int(value)]
            if variable == v and other == u:
                new_domains[other] = [s for s in new_domains[other]
                                      if slot_to_int(s) < slot_to_int(value)]
        if not new_domains[other]:
            return None
    return new_domains


def select_unassigned_variable(assignment, domains, precedence):
    unassigned = [v for v in domains if v not in assignment]
    if not unassigned:
        return None
    # MRV
    min_len = min(len(domains[v]) for v in unassigned)
    candidates = [v for v in unassigned if len(domains[v]) == min_len]
    # Degree heuristic tie-breaker
    if len(candidates) > 1:
        def degree(var):
            return sum(1 for u, v in precedence if u == var or v == var)
        return max(candidates, key=degree)
    return candidates[0]


def order_domain_values(variable, assignment, domains, precedence, latest_slot):
    """LCV — order values by how few neighbour-domain values they eliminate."""
    def eliminated(value):
        count = 0
        for other in domains:
            if other == variable or other in assignment:
                continue
            for val in domains[other]:
                if val == value:
                    count += 1; continue
                if other in latest_slot and slot_to_int(val) > slot_to_int(latest_slot[other]):
                    count += 1; continue
                for u, v in precedence:
                    if variable == u and other == v and slot_to_int(val) <= slot_to_int(value):
                        count += 1; break
                    if variable == v and other == u and slot_to_int(val) >= slot_to_int(value):
                        count += 1; break
        return count
    return sorted(domains[variable], key=eliminated)


def backtracking_search(variables, domains, precedence, latest_slot):
    def backtrack(assignment, cur_domains):
        if len(assignment) == len(variables):
            return dict(assignment)
        var = select_unassigned_variable(assignment, cur_domains, precedence)
        for value in order_domain_values(var, assignment, cur_domains, precedence, latest_slot):
            if is_consistent(var, value, assignment, precedence, latest_slot):
                assignment[var] = value
                nd = forward_check(var, value, assignment, cur_domains, precedence, latest_slot)
                if nd is not None:
                    result = backtrack(assignment, nd)
                    if result is not None:
                        return result
                del assignment[var]
        return None
    return backtrack({}, domains)

In [79]:
def execute_schedule(grid, charging_station, schedule, algorithm="A*"):
    """
    Runs the CSP schedule via graph search.
    Returns (complete_path, total_cost, success).
    """
    current = charging_station
    complete_path = [charging_station]
    cleaned = []
    leg_costs = []

    targets = list(schedule) + [("Charger", charging_station)]

    for label, goal in targets:
        path, cost, nodes = search(grid, current, goal, algorithm)

        if path is None:
            print(f"  [FAILURE] No path from {current} to {label} {goal}.")
            print(f"  Cleaned so far: {cleaned if cleaned else 'none'}")
            return complete_path, path_cost(complete_path), False

        # Validate this leg
        if not validate_path(path, grid):
            print(f"  [FAILURE] Invalid path to {label} {goal}.")
            return complete_path, path_cost(complete_path), False

        complete_path.extend(path[1:])
        leg_costs.append(cost)
        current = goal
        if label != "Charger":
            cleaned.append(label)

    return complete_path, sum(leg_costs), True

## 4. Generic Pipeline

`run_pipeline(env_file, sched_file, algorithm)` runs the **entire** flow end-to-end:

1. Load environment → 2. Load schedule → 3. Solve CSP → 4. Order tasks by slot
→ 5. Execute via search → 6. Print results + final grid.

`run_all_tests()` auto-discovers every `Environment_*.txt` in the folder,
pairs it with the matching `Schedule_*.txt`, and runs all of them.

In [80]:
def _print_grid(grid, title=""):
    if title:
        print(f"  {title}")
    for row in grid:
        print("    " + "  ".join(row))


def run_pipeline(env_file, schedule_file, algorithm="A*", verbose=True):
    """
    Complete pipeline for one (environment, schedule) pair.
    """
    if verbose:
        print("=" * 62)
        print(f"  PIPELINE: {env_file}  +  {schedule_file}   [{algorithm}]")
        print("=" * 62)

    # 1. Load
    try:
        grid, cs, dirty = load_environment(env_file)
        slots, prec, latest = load_schedule(schedule_file)
    except FileNotFoundError as e:
        print(f"  [ERROR] {e}")
        return {"success": False, "reason": f"File not found: {e}"}
    except Exception as e:
        print(f"  [ERROR] Parsing failed: {e}")
        return {"success": False, "reason": f"Parse error: {e}"}

    dirty_map = {f"D{i+1}": d for i, d in enumerate(dirty)}

    if verbose:
        print(f"  Charging station : {cs}")
        print(f"  Dirty locations  : {dirty}")
        print(f"  Time slots       : {slots}")
        print(f"  Precedence       : {prec}")
        print(f"  Latest slots     : {latest}")
        print()
        _print_grid(grid, "BEFORE Cleaning  (initial state — C=Charger, D1..Dn=dirty, X=obstacle):")
        print()

    # 2. CSP
    variables = list(dirty_map.keys())
    if len(variables) > len(slots):
        msg = f"{len(variables)} dirty locations but only {len(slots)} time slots — infeasible."
        if verbose: print(f"  [CSP FAILURE] {msg}")
        return {"success": False, "reason": msg, "grid": grid, "dirty": dirty, "schedule": None}

    domains = {v: slots.copy() for v in variables}
    solution = backtracking_search(variables, domains, prec, latest)

    if solution is None:
        msg = "No feasible schedule exists with the given constraints."
        if verbose: print(f"  [CSP FAILURE] {msg}")
        return {"success": False, "reason": msg, "grid": grid, "dirty": dirty, "schedule": None}

    if verbose:
        print(f"  CSP solution     : {solution}")

    # 3. Order by slot
    order = sorted(solution, key=lambda t: slot_to_int(solution[t]))
    schedule = [(t, dirty_map[t]) for t in order]
    if verbose:
        print(f"  Cleaning order   : {schedule}")

    # 4. Execute
    if verbose:
        print(f"\n  Running robot with {algorithm}...")
    path, cost, ok = execute_schedule(grid, cs, schedule, algorithm)

    if verbose:
        print(f"\n  Total cost       : {cost}")
        print(f"  Success          : {ok}")

        final_grid = [row[:] for row in grid]
        for loc in dirty:
            final_grid[loc[0]][loc[1]] = '.'
        print()
        _print_grid(final_grid, "AFTER Cleaning  (dirty cells turned to '.', robot back at C):")
        print()

    return {"success": ok, "reason": "OK" if ok else "Path not found",
            "solution": solution, "schedule": schedule,
            "path": path, "cost": cost, "grid": grid, "dirty": dirty}


def compare_algorithms(env_file, schedule_file):
    """Run BFS and A* on the same pair and print a comparison table."""
    print("\n" + "=" * 62)
    print(f"  ALGORITHM COMPARISON: {os.path.basename(env_file)} + {os.path.basename(schedule_file)}")
    print("=" * 62)
    rows = []
    for alg in ("BFS", "A*"):
        r = run_pipeline(env_file, schedule_file, alg, verbose=False)
        if r["success"]:
            rows.append((alg, r["cost"], len(r["path"])))
        else:
            rows.append((alg, "FAIL", "FAIL"))
    print(f"  {'Algorithm':<12} {'Cost':<8} {'Path length':<12}")
    print("  " + "-" * 32)
    for alg, cost, plen in rows:
        print(f"  {alg:<12} {cost:<8} {plen:<12}")


def run_all_tests(algorithm="A*", folder=None):
    """Auto-discovers Environment*.txt ↔ Schedule*.txt pairs inside the inputs folder."""
    if folder is None:
        folder = DATA_FOLDER
        
    search_pattern = os.path.join(folder, "Environment*.txt")
    envs = sorted(glob.glob(search_pattern))
    
    pairs = []
    for env in envs:
        base = os.path.basename(env) 
        suffix = base[len("Environment"):-len(".txt")] 
        sched = os.path.join(folder, f"Schedule{suffix}.txt")
        
        if os.path.exists(sched):
            pairs.append((env, sched))
            
    if not pairs:
        print(f"No matching Environment*.txt / Schedule_*.txt pairs found in '{folder}' folder.")
        return
        
    print(f"Discovered {len(pairs)} test pair(s) inside '{folder}':")
    for e, s in pairs:
        print(f"   • {e}  +  {s}")
    print()
    
    results = {}
    for e, s in pairs:
        results[(e, s)] = run_pipeline(e, s, algorithm)
        print()
        
    print("=" * 62)
    print("  SUMMARY")
    print("=" * 62)
    for (e, s), r in results.items():
        status = "✓ SUCCESS" if r["success"] else "✗ FAILED"
        detail = f"cost={r['cost']}" if r["success"] else r.get("reason", "")
        e_name = os.path.basename(e)
        s_name = os.path.basename(s)
        print(f"  {e_name:30s} + {s_name:25s} → {status}  ({detail})")

## 5. How to Use

### Run everything with ONE click
Just click **"Run All"** at the top. 
It will automatically find all `Environment*.txt` and `Schedule*.txt` files in the `inputs` folder, run the pipeline, and display the BFS vs A* comparison at the end.

### To test a new case
Just drop the new files into the `inputs` folder with the naming convention `Environment_Name.txt` and `Schedule_Name.txt`. Then click "Run All" — no code changes needed!

In [81]:
# RUN ONLY ONE SPECIFIC PAIR AT A TIME
# ============================================================

# Ye line sirf wahi file run karegi jo aapne Cell 3 (CONFIG) mein likhi hai
result = run_pipeline(ENV_FILE, SCHEDULE_FILE, ALGORITHM)

# (Optional) Agar aap BFS aur A* ka comparison bhi dekhna chahti hain isi file ka:
print("\n")
compare_algorithms(ENV_FILE, SCHEDULE_FILE)

  PIPELINE: inputs\Environment.txt  +  inputs\Schedule.txt   [A*]
  Charging station : (3, 0)
  Dirty locations  : [(2, 4), (5, 6), (4, 1), (6, 3)]
  Time slots       : ['T1', 'T2', 'T3', 'T4']
  Precedence       : [('D3', 'D1'), ('D1', 'D2'), ('D3', 'D4')]
  Latest slots     : {'D2': 'T3', 'D4': 'T4'}

  BEFORE Cleaning  (initial state — C=Charger, D1..Dn=dirty, X=obstacle):
    .  .  .  .  .  .  .  .
    .  .  .  X  .  .  X  .
    .  .  .  .  D1  .  .  .
    C  .  .  .  .  .  .  .
    .  D3  .  .  X  .  .  .
    .  .  X  .  .  .  D2  .
    X  .  .  D4  .  X  .  .
    .  .  .  .  .  .  .  .

  CSP solution     : {'D1': 'T2', 'D3': 'T1', 'D2': 'T3', 'D4': 'T4'}
  Cleaning order   : [('D3', (4, 1)), ('D1', (2, 4)), ('D2', (5, 6)), ('D4', (6, 3))]

  Running robot with A*...

  Total cost       : 22
  Success          : True

  AFTER Cleaning  (dirty cells turned to '.', robot back at C):
    .  .  .  .  .  .  .  .
    .  .  .  X  .  .  X  .
    .  .  .  .  .  .  .  .
    C  .  .  .  .  